# KG1 V199 conservative continuation

Short 20-step continuation from the submitted V198 final adapter. This notebook trains and gates candidates only; it does not submit to Kaggle.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import hashlib, importlib.util, json, os, pathlib, shutil, subprocess, sys, urllib.request, zipfile
ROOT = pathlib.Path('/content/kg1_v199')
DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/KG1_NVIDIA_V199')
V198_DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/KG1_NVIDIA_V198')
V198_PACK = V198_DRIVE_ROOT / 'kg1_v198_colab_pack.zip'
PACK = V198_PACK if V198_PACK.exists() else DRIVE_ROOT / 'kg1_v198_colab_pack.zip'
PACK_URL = 'https://raw.githubusercontent.com/FELIPEACASTRO/KG1-NVIDIA/31d439bc4a9b33b7b3c772d3526149847103a9b1/runs/v198_micro_distill_colab_pack_20260503/kg1_v198_colab_pack.zip'
PACK_SHA256 = 'e61908c0f75018b0d265c3668600170f6fa99a1a4d559508f489cba9cd6b7c93'
INIT_ADAPTER = V198_DRIVE_ROOT / 'output_v198/final_adapter'
V198_FINAL_ADAPTER_SHA256 = 'dd718b0d416fd9cd6ed928e90e185c131fee9d4cb956f57e59b7d00c3266dafa'
OUT_BASE = DRIVE_ROOT / 'output_v199_conservative_20'

def sha256_path(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def adapter_ready(path):
    cfg = path / 'adapter_config.json'
    model = path / 'adapter_model.safetensors'
    if not cfg.exists() or not model.exists():
        return False
    if cfg.stat().st_size < 100 or model.stat().st_size < 4_000_000_000:
        return False
    json.loads(cfg.read_text(encoding='utf-8'))
    return True

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
assert adapter_ready(INIT_ADAPTER), f'Missing submitted V198 final adapter: {INIT_ADAPTER}'
init_sha = sha256_path(INIT_ADAPTER / 'adapter_model.safetensors')
print('V198 final adapter sha:', init_sha)
assert init_sha == V198_FINAL_ADAPTER_SHA256, 'V198 final adapter SHA mismatch; do not continue from an unverified adapter.'

if not PACK.exists():
    print('V198 pack not found in Drive; downloading verified pack...')
    urllib.request.urlretrieve(PACK_URL, PACK)
pack_hash = sha256_path(PACK)
print('Pack SHA256:', pack_hash)
assert pack_hash == PACK_SHA256, f'Pack SHA mismatch: {pack_hash}'

shutil.rmtree(ROOT, ignore_errors=True)
ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(PACK) as zf:
    zf.extractall(ROOT)
assert (ROOT / 'data/v198/v198_micro_train.strict.jsonl').exists()
assert (ROOT / 'data/v198/v198_micro_val.strict.jsonl').exists()
assert (ROOT / 'scripts/hf_job_train_v90.py').exists()
print('Pack extracted to', ROOT)


In [ ]:
%cd /content/kg1_v199
import importlib.util, os, subprocess, sys
os.environ.setdefault('MAX_JOBS', '4')
os.environ.setdefault('PIP_ROOT_USER_ACTION', 'ignore')

def pip_install(args):
    print('+ pip install', ' '.join(args))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

def pip_uninstall(package_name):
    print('+ pip uninstall -y', package_name)
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', package_name], check=False)

def install_if_missing(module_name, args):
    if importlib.util.find_spec(module_name) is None:
        pip_install(args)
    else:
        print(f'{module_name} already installed')

pip_uninstall('torchao')
pip_install(['--upgrade', 'pip', 'setuptools', 'wheel', 'packaging', 'ninja==1.13.0'])
pip_install(['transformers==5.7.0', 'accelerate==1.13.0', 'peft==0.19.1', 'datasets==4.8.5', 'safetensors==0.7.0', 'huggingface_hub==1.13.0', 'sentencepiece==0.2.1', 'protobuf==7.34.1'])
install_if_missing('causal_conv1d', ['causal-conv1d==1.6.1', '--no-build-isolation'])
install_if_missing('mamba_ssm', ['mamba-ssm==2.3.1', '--no-build-isolation'])
assert importlib.util.find_spec('torchao') is None, 'torchao still installed; restart runtime and rerun cells from top'
import causal_conv1d, mamba_ssm
from mamba_ssm.ops.triton.layernorm_gated import rmsnorm_fn
print('mamba_ssm OK:', getattr(mamba_ssm, '__version__', 'unknown'))


In [ ]:
import subprocess
gpu = subprocess.check_output('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader', shell=True).decode().strip()
print(gpu)
assert ('H100' in gpu or 'A100' in gpu), 'Use H100 HighRAM or A100 HighRAM for this run.'


In [ ]:
import os, pathlib, shutil, urllib.request
OUT = OUT_BASE
if OUT.exists():
    import datetime
    suffix = datetime.datetime.utcnow().strftime('%Y%m%d_%H%M%S')
    OUT = pathlib.Path(str(OUT_BASE) + '_' + suffix)
OUT.mkdir(parents=True, exist_ok=True)
print('V199_OUT =', OUT)
FIXED_TRAIN_SCRIPT_URL = 'https://raw.githubusercontent.com/FELIPEACASTRO/KG1-NVIDIA/31d439bc4a9b33b7b3c772d3526149847103a9b1/scripts/hf_job_train_v90.py'
TRAIN_SCRIPT = pathlib.Path('/content/kg1_v199/scripts/hf_job_train_v90.py')
script_text = TRAIN_SCRIPT.read_text(encoding='utf-8') if TRAIN_SCRIPT.exists() else ''
if 'load_peft_weights_with_direct_fallback' not in script_text:
    print('Runtime has stale hf_job_train_v90.py; downloading PEFT direct-load fixed script...')
    urllib.request.urlretrieve(FIXED_TRAIN_SCRIPT_URL, TRAIN_SCRIPT)
script_text = TRAIN_SCRIPT.read_text(encoding='utf-8')
assert 'load_peft_weights_with_direct_fallback' in script_text
assert 'PEFT_MANUAL_LOAD_METHOD' in script_text
os.environ['UPLOAD_TO_HF'] = '0'
os.environ['MODEL_NAME'] = 'nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16'
os.environ['DATA_FILE'] = '/content/kg1_v199/data/v198/v198_micro_train.strict.jsonl'
os.environ['VAL_FILE'] = '/content/kg1_v199/data/v198/v198_micro_val.strict.jsonl'
os.environ['INIT_ADAPTER_DIR'] = str(INIT_ADAPTER)
os.environ['INIT_ADAPTER_LOAD_MODE'] = 'manual'
os.environ['PEFT_MANUAL_LOAD_METHOD'] = 'direct'
os.environ['OUTPUT_DIR'] = str(OUT)
os.environ['V199_OUT'] = str(OUT)
os.environ['RUN_ID'] = 'v199-conservative-v198-final-20s'
os.environ['MAX_LENGTH'] = '2048'
os.environ['BATCH_SIZE'] = '16'
os.environ['MICRO_BATCH_SIZE'] = '1'
os.environ['GRADIENT_CHECKPOINTING'] = '1'
os.environ['MAX_STEPS'] = '20'
os.environ['SAVE_EVERY_STEPS'] = '10'
os.environ['EVAL_EVERY_STEPS'] = '10'
os.environ['EVAL_MAX_EXAMPLES'] = '360'
os.environ['LEARNING_RATE'] = '3e-6'
os.environ['FINAL_LEARNING_RATE'] = '8e-7'
os.environ['ABORT_EVAL_LOSS_GT'] = '0.98'
os.environ['EXPECTED_TRAIN_SHA256'] = '6d2742616300818eb50c54d36019551b24f5b71c607a2b28feda7461a709def0'
os.environ['EXPECTED_VAL_SHA256'] = 'e59c907c6545e5e587097a64762e3e874508e8cd74d85d5c7c79354ebe56e73c'
os.environ['MIN_TRAIN_EXAMPLES'] = '1875'
os.environ['MIN_TOKENIZED_TRAIN_EXAMPLES'] = '1600'
os.environ['MIN_VAL_EXAMPLES'] = '720'
os.environ['MIN_TOKENIZED_VAL_EXAMPLES'] = '700'
os.environ['TRAINABLE_LORA_MODULES'] = 'in_proj,out_proj,q_proj,k_proj,v_proj,o_proj'
os.environ['MAX_TRAINABLE_PARAM_RATIO'] = '0.035'
!python scripts/hf_job_train_v90.py


Convert and gate the V199 adapters. This still does not submit to Kaggle.


In [ ]:
import json, os, pathlib, subprocess, sys, urllib.request
BASE = 'https://raw.githubusercontent.com/FELIPEACASTRO/KG1-NVIDIA/claude/competent-shamir/scripts'
for name in ['kg1_v198_posttrain_gate.py', 'kg1_v199_posttrain_gate.py', 'nemotron_submission_preflight.py', 'kg1_submission_gate.py', 'kg1_v198_final_submit_doublecheck.py']:
    dst = pathlib.Path('/content/kg1_v199/scripts') / name
    print('downloading', name)
    urllib.request.urlretrieve(f'{BASE}/{name}', dst)

!python scripts/kg1_v199_posttrain_gate.py --root /content/kg1_v199 --output-root "$V199_OUT" --fail-on-block

report_path = pathlib.Path(os.environ['V199_OUT']) / 'posttrain_kaggle_gate/v199_posttrain_gate_report.json'
report = json.loads(report_path.read_text(encoding='utf-8'))
assert report['decision']['ready'], report['decision']
primary_zip = report['decision']['primary_zip']
primary_label = report['decision']['primary_label']
print('primary_label =', primary_label)
print('primary_zip =', primary_zip)

preflight_json = pathlib.Path(os.environ['V199_OUT']) / f'{primary_label}_preflight.json'
subprocess.run([sys.executable, 'scripts/nemotron_submission_preflight.py', '--adapter-zip', primary_zip, '--output-json', str(preflight_json), '--fail-on-block'], check=True)

doublecheck_json = pathlib.Path(os.environ['V199_OUT']) / f'{primary_label}_submit_doublecheck.json'
subprocess.run([
    sys.executable, 'scripts/kg1_v198_final_submit_doublecheck.py',
    '--candidate-zip', primary_zip,
    '--expected-label', 'final' if primary_label == 'final' else 'checkpoint30',
    '--posttrain-report', str(report_path),
    '--preflight-report', str(preflight_json),
    '--output-json', str(doublecheck_json),
    '--fail-on-block',
], check=True)
print('V199 gated candidate ready. No Kaggle submit was performed.')
print('doublecheck:', doublecheck_json)
